In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.blob import ContainerClient

TENANT_ID = "5a13decf-4ae8-423c-9d6f-320db1def1ad"
CLIENT_ID = "6fcdb218-bf19-483e-95ab-b86fb024790a"
secret = dbutils.secrets.get(catalog="workspace", schema="default", key="azure-reader-secret")

credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, secret)
container = ContainerClient("https://stpowermarketfm.blob.core.windows.net", "raw", credential=credential)

In [0]:
names = [b.name for b in container.list_blobs(name_starts_with="consumption/") if b.name.endswith(".json")]
print(len(names))
print(names[:3])

14
['consumption/2026-10-05/consumption_2026-10-05.json', 'consumption/2026-10-06/consumption_2026-10-06.json', 'consumption/2026-10-07/consumption_2026-10-07.json']


In [0]:
import json
raw = json.loads(container.download_blob(names[0]).readall())
for item in raw["data"]:
    print(item["id"], len(item["attributes"]["consumptionPerGroupMbaHour"]))

* 0
NO1 120
NO2 120
NO3 120
NO4 120
NO5 120


In [0]:
from datetime import datetime, timezone

rows = []
loaded_at = datetime.now(timezone.utc).isoformat()
for name in names:
    raw = json.loads(container.download_blob(name).readall())
    for item in raw["data"]:
        for r in item["attributes"]["consumptionPerGroupMbaHour"]:
            r["source_file"] = name
            r["loaded_at"] = loaded_at
            rows.append(r)
print(len(rows))

8400


In [0]:
df = spark.createDataFrame(rows)
df.write.mode("overwrite").saveAsTable("workspace.default.bronze_consumption")
print(spark.table("workspace.default.bronze_consumption").count())

8400


In [0]:
df.select("startTime", "priceArea", "consumptionGroup", "quantityKwh").show(3, truncate=False)

+-------------------------+---------+----------------+-----------+
|startTime                |priceArea|consumptionGroup|quantityKwh|
+-------------------------+---------+----------------+-----------+
|2026-10-05T00:00:00+02:00|NO1      |cabin           |57659.516  |
|2026-10-05T01:00:00+02:00|NO1      |cabin           |56477.89   |
|2026-10-05T02:00:00+02:00|NO1      |cabin           |55733.47   |
+-------------------------+---------+----------------+-----------+
only showing top 3 rows
